In [1]:
import json
from sklearn.model_selection import train_test_split
import re
from tqdm import tqdm
from datasets import Dataset
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import shap
import pickle
import string
from collections import defaultdict
import numpy as np
from glob import glob

In [2]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

print(project_root)

/home/imruhi/Documents/RiskPerceptionSeafare


In [3]:
from data_gathering.utils.clean_text import clean_text
from classification.utils_finetune import load_dataset, split_dataset

In [4]:
import logging
logging.getLogger('shap').setLevel(logging.WARNING) # turns off the "shap INFO" logs
logging.getLogger('matplotlib').setLevel(logging.WARNING) # turns off the progress bar

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
with open("/home/imruhi/Documents/RiskPerceptionSeafare/params.json", 'r') as f:
    PARAMS = json.load(f)

In [7]:
model_id = PARAMS["classi_finetune_model"]
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

Cleaning text


  0%|          | 0/5859 [00:00<?, ?it/s]

100%|██████████| 5859/5859 [00:00<00:00, 185254.86it/s]

Size of dataset: 5859
{'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
{0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
Train: Counter({1: 2089, 2: 1662, 0: 936})
Test: Counter({1: 261, 2: 208, 0: 117})
Val: Counter({1: 261, 2: 208, 0: 117})


In [8]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [9]:
for l in dataset["label"].unique():
    print(l)
    subset = dataset[dataset["label"]==l]
    all_texts = " ".join(subset["text"])
    all_texts = all_texts.lower()
    all_texts = all_texts.translate(str.maketrans('', '', string.punctuation))
    all_texts = all_texts.split(" ")
    all_texts = [w for w in all_texts if w not in stop_words and w!=""]
    counter = Counter(all_texts)
    print(counter.most_common(20))
    print()

HIGH
[('alexandria', 683), ('city', 198), ('egypt', 167), ('also', 155), ('bishop', 150), ('one', 138), ('called', 129), ('time', 113), ('sea', 106), ('great', 106), ('king', 92), ('first', 90), ('ptolemy', 87), ('two', 83), ('island', 82), ('alexander', 79), ('son', 76), ('church', 75), ('years', 74), ('many', 71)]

MEDIUM
[('rome', 1416), ('city', 716), ('year', 330), ('also', 300), ('one', 269), ('time', 267), ('called', 261), ('first', 232), ('son', 218), ('years', 195), ('consuls', 186), ('name', 174), ('athens', 173), ('two', 163), ('made', 163), ('temple', 161), ('sea', 160), ('lucius', 158), ('king', 156), ('great', 155)]

LOW
[('alexandria', 688), ('city', 365), ('also', 298), ('sea', 279), ('called', 275), ('one', 270), ('island', 233), ('river', 226), ('name', 192), ('time', 180), ('egypt', 171), ('great', 159), ('bishop', 149), ('many', 143), ('place', 140), ('near', 139), ('two', 137), ('first', 128), ('promontory', 117), ('coast', 113)]



In [10]:
dataset["label"].value_counts()

label
MEDIUM    2611
LOW       2078
HIGH      1170
Name: count, dtype: int64

In [11]:
len(dataset)

5859

In [12]:
val_data.to_pandas().to_csv("val_data.csv")

In [13]:
# Load your fine-tuned BERT model and tokenizer
model_path = f'{PARAMS["save_model"]}{model_id.split("/")[-1]}_finetuned_16_12'
model_base = PARAMS["classi_finetune_model"]
tokenizer = AutoTokenizer.from_pretrained(model_base)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model = model.to("cuda")
model.eval()

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(256000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (a

In [14]:
preds = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
)


In [15]:
explainer = shap.Explainer(
    preds, seed=42
)

In [16]:
texts = val_data["text"]

In [17]:
size = 10

In [18]:
# get shap values in batches

# shap_values = []
# for itr in range(0, len(texts), size):
#     print(f"Processing batch {itr} - {itr+size}")
#     batch_texts = texts[itr:itr+size]
#     batch_shap_values = explainer(batch_texts)
#     with open(f"shap_batch_{itr}.pkl", "wb") as f:
#         pickle.dump(batch_shap_values, f)
#     shap_values.append(batch_shap_values)

In [19]:
def get_sorted_shap_files(path_pattern="shap_values/shap_batch_*.pkl"):
    files = glob(path_pattern)

    def extract_number(f):
        match = re.search(r"shap_batch_(\d+)\.pkl", f)
        return int(match.group(1)) if match else -1

    return sorted(files, key=extract_number)

In [20]:
files = get_sorted_shap_files()

In [21]:

all_data = []
all_class_shap = []

for i in files:
    with open(i, "rb") as f:
        batch_sv = pickle.load(f)
    all_class_shap.extend(batch_sv.values)

# to keep the _ token (to join words back)            
for itr in range(0, len(texts), size):
    batch_texts = texts[itr:itr+size]
    for x in batch_texts:
        all_data.append(tokenizer.tokenize(x))


In [22]:
len(all_class_shap), len(all_data), len(texts)

(586, 586, 586)

In [23]:
def merge_tokens(tokens, values):
    merged_tokens = []
    merged_values = []

    current_token = ""
    current_value = None

    for t, v in zip(tokens, values):

        # RoBERTa word start token
        if t.startswith("▁") and t not in string.punctuation:

            # flush previous token
            if current_token != "":
                merged_tokens.append(current_token)
                merged_values.append(current_value)

            current_token = t[1:]  # remove _
            current_value = v.copy()

        else:
            if t not in string.punctuation:
                # continuation of same word
                current_token += t
                current_value += v

    # flush last token
    if current_token:
        merged_tokens.append(current_token)
        merged_values.append(current_value)

    return merged_tokens, np.array(merged_values)

In [24]:
# def group_values(tokens, values, group=2):
#     paired_tokens = []
#     paired_values = []

#     for i in range(0, len(tokens), group):
#         pair_tokens = tokens[i-group:i+group]
#         pair_values = values[i-group:i+group]

#         # join tokens with space
#         paired_tokens.append(" ".join(pair_tokens))

#         # combine values (sum, mean, etc.)
#         paired_values.append(np.sum(pair_values, axis=0))

#     return paired_tokens, np.array(paired_values)

In [25]:
class_contribs = [defaultdict(list) for _ in range(3)]
class_contexts = [defaultdict(list) for _ in range(3)]
group = 20
for d, sv in zip(all_data, all_class_shap):
    merged_tokens, merged_values = merge_tokens(d, sv)
    # merged_tokens, merged_values = group_values(merged_tokens, merged_values, group)
    for i, token in enumerate(merged_tokens):
        for c in range(3):
            # get context around max value words 
            context = " ".join((
                            merged_tokens[max(i - group, 0):i]
                            + [f"[[{token}]]"]
                            + merged_tokens[i + 1:i + group + 1]
                    ))
            
            class_contribs[c][token].append(merged_values[i, c])
            class_contexts[c][token].append(context)          
            

In [32]:
len(class_contribs[0])

6085

In [26]:
top_k = 10

agg = []

for c in range(3):
    token_scores = {
        token: np.mean(vals)
        for token, vals in class_contribs[c].items()
    }
    agg.append(token_scores)
    

for c in range(3):

    print(f"\nTop words for class {id2label[c]}:")

    sorted_tokens = sorted(
        [(token, score) for token, score in agg[c].items() if score > 0],
        key=lambda x: x[1],
        reverse=True
    )

    for token, score in sorted_tokens[:top_k]:

        # choose one example context
        example_context = class_contexts[c][token][:3]

        print(f"\n{token}: {score:.4f}")
        examples = "\n    ".join(example_context)
        print(f"    {examples}")


Top words for class HIGH:

Μασσαλία: 0.2282
    [[Μασσαλία]] Massalia Marseille Isocrates says in Archidamos that Phokaians having fled the despotic rule of the Great King

(Corsica).: 0.2281
    Off the Tyrrhenian coast is the island of Kyrnos [[(Corsica).]] From Tyrrhenia the voyage to Kyrnos is a day and a half There is an inhabited island in the

Agrippinus: 0.2048
    Lucius Caesar celebrated a triumph with his brother over the Parthians [[Agrippinus]] presided as the 9th bishop of Alexandria for 12 years

Isocrates: 0.1847
    Μασσαλία Massalia Marseille [[Isocrates]] says in Archidamos that Phokaians having fled the despotic rule of the Great King

Charibael: 0.1819
    as Alexandria now receives the things brought both from abroad and from Egypt But not long before our own time [[Charibael]] destroyed the place

boys: 0.1711
    deserves our fostering care it is the sacred art of music Do you therefore select from the citizens of Alexandria [[boys]] of good birth and give orde

In [27]:
top_k = 10

agg = []

for c in range(3):
    token_scores = {
        token: np.mean(vals)
        for token, vals in class_contribs[c].items()
    }
    agg.append(token_scores)
    

for c in range(3):

    print(f"\nTop words for class {id2label[c]}:")

    sorted_tokens = sorted(
        [(token, score) for token, score in agg[c].items() if score < 0],
        key=lambda x: x[1],
        reverse=False
    )

    for token, score in sorted_tokens[:top_k]:

        # choose one example context
        example_context = class_contexts[c][token][:3]

        print(f"\n{token}: {score:.4f}")
        examples = "\n    ".join(example_context)
        print(f"    {examples}")


Top words for class HIGH:

Ionia: -0.0582
    of the Massiliots and is larger than the one inside the temple The Massiliots are a colony of Phocaea in [[Ionia]] and their city was founded by some of those who ran away

Minor: -0.0544
    From Makaraia to Sabratha 400 stades it is a city without a harbor It has an open roadstead Syrtis [[Minor]] follows

consul.,: -0.0456
    While these events were occurring in Italy the [[consul.,]] Cn Servilius Geminus with a fleet of 120 vessels visited Sardinia and Corsica and received hostages from both islands from

Cherronesos: -0.0432
    — for it is drinkable — it is a brief sail upstream to the lake from Pharos There is also [[Cherronesos]] with a harbor the coastal voyage is 200

Peter: -0.0390
    Tyrannus is appointed as the 19th bishop of Antioch After Theonus [[Peter]] is ordained the 21st bishop of the Church of Alexandria who later in the ninth year of
    to God and has been dedicated to him the monk One must note that when Claudius 